178992 Ane Kitanoska
Lot Sizing

Instance Loader

In [55]:
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

with open(filename, "r") as file:
    data = file.read()

print("Loaded:", filename)

Saving ls-20-idle.dzn to ls-20-idle.dzn
Loaded: ls-20-idle.dzn


Parameter Declaration

In [56]:
def get_value(data, name):
    for part in data.split(";"):
        if name in part:
            return part.split("=")[1].strip()


def get_list(data, name):
    for part in data.split(";"):
        if name in part:
            values = part.split("=")[1].strip()
            values = values.replace("[", "").replace("]", "")
            return [int(x) for x in values.split(",")]


def get_matrix(data, name):
    for part in data.split(";"):
        if name in part:
            matrix = part.split("=")[1].strip()
            matrix = matrix.replace("[|", "")
            matrix = matrix.replace("|]", "")
            rows = matrix.split("|")

            return [[int(x) for x in row.split(",")] for row in rows]


Periods = int(get_value(data, "Periods"))
Items = int(get_value(data, "Items"))

Demands = get_matrix(data, "Demands")
StockingCosts = get_list(data, "StockingCosts")
SetupCosts = get_matrix(data, "SetupCosts")


print("Periods:", Periods)
print("Items:", Items)
print("Demands:", Demands)
print("Stocking Costs:", StockingCosts)
print("Setup Costs:", SetupCosts)

Periods: 20
Items: 4
Demands: [[0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0], [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0]]
Stocking Costs: [14, 20, 10, 10]
Setup Costs: [[0, 196, 103, 160], [170, 0, 189, 183], [115, 180, 0, 194], [164, 161, 104, 0]]


Decision Variable

In [57]:
ProductionPlan = [0] * Periods

Feasibility

In [58]:
def is_feasible(ProductionPlan):

    for i in range(Items):

        production = 0
        demand = 0

        for p in range(Periods):

            if ProductionPlan[p] == i + 1:
                production += 1

            demand += Demands[i][p]

            if production < demand:
                return False

        if production != demand:
            return False

    return True

Inventory

In [59]:
def calculate_inventory(ProductionPlan):

    Inventory = []

    for i in range(Items):
        item_inventory = []
        stock = 0

        for p in range(Periods):

            if ProductionPlan[p] == i + 1:
                stock += 1

            stock -= Demands[i][p]

            item_inventory.append(stock)

        Inventory.append(item_inventory)

    return Inventory

Inventory = calculate_inventory(ProductionPlan)

Stocking Cost

In [60]:
def calculate_stocking_cost(Inventory):

    total_cost = 0

    for i in range(Items):
        for p in range(Periods):
            total_cost += Inventory[i][p] * StockingCosts[i]

    return total_cost

StockingCost = calculate_stocking_cost(Inventory)

Setup Costs

In [61]:
def calculate_setup_cost(ProductionPlan):

    total_cost = 0
    previous_item = 0

    for p in range(Periods):

        current_item = ProductionPlan[p]

        if current_item != 0:

            if previous_item != 0:
                total_cost += SetupCosts[
                    previous_item - 1
                ][
                    current_item - 1
                ]

            previous_item = current_item

    return total_cost

SetupCost = calculate_setup_cost(ProductionPlan)

Remaining Demand

In [62]:
def remaining_demand(ProductionPlan, item):

    produced = ProductionPlan.count(item)
    demanded = sum(Demands[item - 1])

    return demanded - produced

Next Demand

In [63]:
def next_demand_period(ProductionPlan, item):

    produced = 0

    for p in range(Periods):

        if ProductionPlan[p] == item:
            produced += 1

        if Demands[item - 1][p] == 1:
            if produced == 0:
                return p
            else:
                produced -= 1

    return Periods

Greedy

In [64]:
def greedy():

    ProductionPlan = [0] * Periods

    for p in range(Periods):

        best_item = 0
        best_period = Periods

        for item in range(1, Items + 1):

            if remaining_demand(ProductionPlan, item) > 0:

                demand_period = next_demand_period(
                    ProductionPlan, item
                )

                if demand_period < best_period:

                    best_period = demand_period
                    best_item = item

        ProductionPlan[p] = best_item

    return ProductionPlan

Run Greedy

In [65]:
ProductionPlan = greedy()

Inventory = calculate_inventory(ProductionPlan)

StockingCost = calculate_stocking_cost(Inventory)

SetupCost = calculate_setup_cost(ProductionPlan)

TotalCost = StockingCost + SetupCost

IdlePeriods = ProductionPlan.count(0)

Results from Greedy

In [66]:
print("Greedy Production Plan:", ProductionPlan)
print("Feasible:", is_feasible(ProductionPlan))
print("Inventory:", Inventory)
print("Stocking Cost:", StockingCost)
print("Setup Cost:", SetupCost)
print("Total Cost:", TotalCost)
print("Idle Periods:", IdlePeriods)

Greedy Production Plan: [2, 2, 3, 4, 1, 3, 1, 3, 1, 1, 4, 4, 1, 1, 4, 3, 0, 0, 0, 0]
Feasible: True
Inventory: [[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 2, 2, 1, 0, 0, 0, 0], [1, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0], [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0]]
Stocking Cost: 378
Setup Cost: 1571
Total Cost: 1949
Idle Periods: 4


Improved Greedy

In [67]:
def get_previous_item(ProductionPlan, period):
    previous_item = 0

    for p in range(period - 1, -1, -1):
        if ProductionPlan[p] != 0:
            previous_item = ProductionPlan[p]
            break

    return previous_item

def greedy_improved():
    ProductionPlan = [0] * Periods

    for p in range(Periods):

        best_item = 0
        best_period = Periods
        best_setup = None

        previous_item = get_previous_item(ProductionPlan, p)

        for item in range(1, Items + 1):

            if remaining_demand(ProductionPlan, item) > 0:

                demand_period = next_demand_period(
                    ProductionPlan, item
                )

                setup_cost = 0

                if previous_item != 0:
                    setup_cost = SetupCosts[
                        previous_item - 1
                    ][
                        item - 1
                    ]

                # First priority: earliest demand
                if demand_period < best_period:

                    best_period = demand_period
                    best_item = item
                    best_setup = setup_cost

                # Second priority: lowest setup cost
                elif demand_period == best_period:

                    if best_setup is None or setup_cost < best_setup:
                        best_item = item
                        best_setup = setup_cost

        ProductionPlan[p] = best_item

    return ProductionPlan

Run Improved Greedy

In [68]:
ProductionPlanImproved = greedy_improved()

InventoryImproved = calculate_inventory(ProductionPlanImproved)
StockingCostImproved = calculate_stocking_cost(InventoryImproved)
SetupCostImproved = calculate_setup_cost(ProductionPlanImproved)
TotalCostImproved = StockingCostImproved + SetupCostImproved
IdlePeriodsImproved = ProductionPlanImproved.count(0)

Results Improved Greedy

In [69]:
print()
print("Improved Greedy Production Plan:", ProductionPlanImproved)
print("Feasible:", is_feasible(ProductionPlanImproved))
print("Inventory:", InventoryImproved)
print("Stocking Cost:", StockingCostImproved)
print("Setup Cost:", SetupCostImproved)
print("Total Cost:", TotalCostImproved)
print("Idle Periods:", IdlePeriodsImproved)


Improved Greedy Production Plan: [2, 2, 4, 3, 1, 3, 3, 1, 1, 1, 4, 4, 1, 1, 4, 3, 0, 0, 0, 0]
Feasible: True
Inventory: [[0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 2, 2, 1, 0, 0, 0, 0], [1, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 1, 1, 2, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0], [0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0]]
Stocking Cost: 374
Setup Cost: 1208
Total Cost: 1582
Idle Periods: 4


Backtracking

In [70]:
best_plan = None
best_cost = float("inf")


def backtracking(ProductionPlan, period):

    global best_plan, best_cost

    # Check for backlog


    for i in range(Items):

        production = 0
        demand = 0

        for p in range(period):

            if ProductionPlan[p] == i + 1:
                production += 1

            demand += Demands[i][p]

        if production < demand:
            return

    # Calculate current cost


    Inventory = calculate_inventory(ProductionPlan)

    current_inventory_cost = 0

    for i in range(Items):
        for p in range(period):
            current_inventory_cost += (
                Inventory[i][p] * StockingCosts[i]
            )

    current_setup_cost = 0
    previous_item = 0

    for p in range(period):

        current_item = ProductionPlan[p]

        if current_item != 0:

            if previous_item != 0:
                current_setup_cost += SetupCosts[
                    previous_item - 1
                ][
                    current_item - 1
                ]

            previous_item = current_item

    current_cost = current_inventory_cost + current_setup_cost


    # Branch and Bound

    if current_cost > best_cost:
        return


    # Complete schedule

    if period == Periods:

        if not is_feasible(ProductionPlan):
            return

        if current_cost < best_cost:

            best_plan = ProductionPlan.copy()
            best_cost = current_cost

        return


    # Try all possibilities
    # 0 = idle
    # 1...Items = production


    for item in range(Items + 1):

        ProductionPlan[period] = item

        backtracking(ProductionPlan, period + 1)

        ProductionPlan[period] = 0

Run Backtracking

In [71]:
best_plan = None
best_cost = float("inf")

ProductionPlan = [0] * Periods

backtracking(ProductionPlan, 0)

print()

print("Backtracking Production Plan:", best_plan)
print("Feasible:", is_feasible(best_plan))

Inventory = calculate_inventory(best_plan)

StockingCost = calculate_stocking_cost(Inventory)
SetupCost = calculate_setup_cost(best_plan)


Backtracking Production Plan: [2, 2, 4, 3, 3, 3, 1, 1, 1, 1, 4, 0, 4, 4, 3, 1, 1, 0, 0, 0]
Feasible: True


Results from Backtracking

In [72]:
print("Inventory:", Inventory)
print("Stocking Cost:", StockingCost)
print("Setup Cost:", SetupCost)
print("Total Cost:", StockingCost + SetupCost)
print("Idle Periods:", best_plan.count(0))

Inventory: [[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 2, 2, 2, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0], [0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0]]
Stocking Cost: 292
Setup Cost: 781
Total Cost: 1073
Idle Periods: 4
